In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Fungsi Qiskit adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API IBM Quantum Platform). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.


<Accordion>
<AccordionItem title="Package versions">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>

## Gambaran keseluruhan
Dalam kimia kuantum, masalah struktur elektronik memfokuskan pada mencari penyelesaian kepada persamaan Schrödinger elektronik — fungsi gelombang kuantum yang menggambarkan tingkah laku elektron sistem. Fungsi gelombang ini adalah vektor amplitud kompleks, dengan setiap amplitud sepadan dengan sumbangan konfigurasi elektron yang mungkin.

Keadaan dasar adalah fungsi gelombang tenaga terendah sistem dan mempunyai kepentingan istimewa dalam kajian sistem molekul. Pendekatan paling tepat untuk mengira keadaan dasar mempertimbangkan semua konfigurasi elektron yang mungkin, tetapi ini menjadi tidak dapat diuruskan untuk sistem yang lebih besar kerana bilangan konfigurasi bertumbuh secara eksponen dengan saiz sistem.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) adalah kaedah hibrid kuantum-klasik yang inovatif untuk menganggar keadaan dasar sistem molekul dengan tepat. Ia mengintegrasikan perkakasan kuantum dengan pengkomputeran klasik, menggunakan pemproses kuantum untuk meneroka konfigurasi elektron calon dengan cekap dan mengira fungsi gelombang yang terhasil pada komputer klasik. Dengan menghasilkan fungsi gelombang yang padat namun tepat secara kimia, HI-VQE meningkatkan penyelidikan dan penemuan dalam kimia kuantum dan sains bahan.

![Image showing an overview of Qunova's HI-VQE algorithm](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE mengurangkan kerumitan pengkomputeran masalah struktur elektronik dengan menganggar keadaan dasar secara cekap dengan ketepatan tinggi. Ia memfokuskan pada subset konfigurasi elektron yang paling relevan yang dipilih dengan teliti, mengoptimumkan ketepatan dan kecekapan.

Menggabungkan kekuatan komputer klasik dan kuantum, HI-VQE secara berulang memperhalusi dan menambah baik anggaran fungsi gelombang semasa. Teknik pembinaan subspace yang unik membantu menjadikan pemilihan konfigurasi lebih cekap, supaya pengguna mempunyai kawalan pengkomputeran yang lebih besar dan ketepatan yang lebih baik dalam simulasi kimia kuantum.

Jika anda ingin mempelajari algoritma dengan lebih mendalam, anda boleh [membaca kertas penyelidikan berkaitan](https://arxiv.org/abs/2503.06292).
## Penerangan
Bilangan konfigurasi elektron untuk sistem molekul bertumbuh secara eksponen dengan saiz sistem. Walau bagaimanapun, untuk keadaan elektronik tertentu, seperti keadaan dasar, adalah lazim bahawa hanya sebahagian kecil konfigurasi menyumbang secara signifikan kepada tenaga keadaan. Kaedah Interaksi Konfigurasi Terpilih (SCI) memanfaatkan sifat jarang ini untuk mengurangkan kos pengkomputeran dengan mengenal pasti dan memfokuskan pada konfigurasi yang paling relevan. Subset konfigurasi ini dirujuk sebagai subspace.

HI-VQE memanfaatkan kecekapan intrinsik komputer kuantum untuk mewakili sistem molekul bagi membantu pencarian subspace. Ia mengintegrasikan subrutin klasik dan kuantum untuk menyelesaikan masalah struktur elektronik dengan ketepatan tinggi. Berbeza dengan kaedah SCI kuantum sedia ada, HI-VQE menggabungkan latihan variasi, pembinaan subspace berulang, dan penapisan konfigurasi pra-pendiagonalan untuk meningkatkan kecekapan dengan mengurangkan pengukuran kuantum, lelaran, dan kos pendiagonalan klasik. Oleh itu, HI-VQE boleh diaplikasikan kepada sistem molekul yang lebih besar yang memerlukan lebih banyak qubit, dan mengurangkan kos untuk menyelesaikan masalah bersaiz tertentu kepada tahap ketepatan yang sama.

![Image showing a detailed description of each step in Qunova's HI-VQE algorithm.](../docs/images/guides/qunova-chemistry/description.avif)

Untuk mengira keadaan dasar sistem, HI-VQE mula-mula menggunakan pakej kimia klasik PySCF untuk menjana perwakilan molekul daripada input yang diberikan pengguna, seperti geometri molekul dan maklumat molekul lain. Ia kemudian memasuki gelung pengoptimuman hibrid kuantum-klasik, secara berulang memperhalusi subspace untuk mewakili keadaan dasar secara optimum sambil meminimumkan bilangan konfigurasi yang disertakan. Gelung berterusan sehingga kriteria penumpuan, seperti saiz subspace atau kestabilan tenaga, dipenuhi, selepas itu fungsi gelombang keadaan dasar dan tenaga yang dikira dikeluarkan. Keputusan ini boleh digunakan untuk membina permukaan tenaga potensial yang tepat dan melakukan analisis lanjutan sistem.

Gelung pengoptimuman memfokuskan pada melaraskan parameter litar kuantum untuk menghasilkan subspace berkualiti tinggi. HI-VQE menawarkan tiga pilihan litar kuantum: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), dan [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Pengoptimuman dimulakan dekat dengan keadaan rujukan Hartree-Fock kerana kesesuaiannya yang umum. litar kemudian dilaksanakan pada peranti kuantum dan konfigurasi disampling dari keadaan kuantum yang terhasil sebelum dikembalikan sebagai rentetan binari. Disebabkan hingar peranti kuantum, beberapa konfigurasi yang disampling mungkin tidak sah secara fizikal, gagal memulihara bilangan elektron atau spin. HI-VQE menangani ini menggunakan proses pemulihan konfigurasi dari pakej [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), supaya pengguna boleh sama ada membetulkan konfigurasi tidak sah atau membuangnya.

Konfigurasi yang sah kemudian melalui langkah penapisan pilihan untuk mengalih keluar yang dijangka menyumbang secara minimal. Ini mengurangkan dimensi subspace, seterusnya menurunkan kos langkah pendiagonalan. Jika penapisan diaktifkan, Hamiltonian subspace awal dibina dari konfigurasi yang sah dan pendiagonalan dilakukan dengan kriteria penamatan yang sangat longgar. Walaupun ketepatan amplitud yang terhasil untuk setiap konfigurasi adalah rendah, ia berkesan untuk meramalkan konfigurasi mana yang hendak ditinggalkan daripada subspace pada lelaran ini, dan ia cepat untuk dikira.

Konfigurasi yang dipilih ditambahkan ke subspace, dan Hamiltonian sistem diprojeksikan ke dalam subspace ini. Subspace dikemas kini secara berulang, mengekalkan konfigurasi yang paling relevan merentasi lelaran. Pendekatan ini berbeza dengan kaedah alternatif kerana litar kuantum tidak perlu menghampiri keadaan dasar penuh pada setiap langkah.

Seterusnya, Hamiltonian subspace didiagonalkan secara klasik untuk mendapatkan nilai eigen terendah dan eigenvector yang sepadan, mewakili penghampiran keadaan dasar dan tenaganya. Apabila kualiti subspace bertambah baik melalui lelaran, keadaan dasar yang dikira lebih menghampiri keadaan dasar sebenar. Langkah penapisan tambahan boleh dilakukan pada ketika ini untuk mengalih keluar konfigurasi daripada subspace yang tidak mempunyai sumbangan ketara kepada keadaan dasar yang dikira. Langkah ini memastikan subspace yang dibawa ke lelaran seterusnya adalah sepadat mungkin. Ini dinilai berdasarkan amplitud yang dikembalikan oleh pendiagonalan, kerana ini mewakili kepentingan sumbangan setiap konfigurasi kepada keadaan dasar yang dikira.

Semakan penumpuan kemudian menentukan sama ada latihan lanjut akan meningkatkan keputusan. Jika ya, langkah pengembangan klasik pilihan dilakukan, parameter litar kuantum dikemas kini untuk lebih meminimumkan tenaga yang dikira, dan proses berulang. Langkah pengembangan klasik menjana konfigurasi tambahan untuk subspace, melengkapi konfigurasi yang disampling dari peranti kuantum. Ia mula-mula mengenal pasti konfigurasi dengan amplitud terbesar dalam keputusan pendiagonalan, sebelum menjana konfigurasi baharu dengan eksitasi tunggal dan berganda dari konfigurasi yang dikenal pasti. Bilangan konfigurasi yang dikehendaki kemudian ditambahkan ke subspace.

Setelah ditentukan bahawa lelaran telah menumpu, HI-VQE mengembalikan keadaan dasar yang dikira (dalam bentuk keadaan dalam subspace dan amplitud mereka dalam fungsi gelombang keadaan dasar), tenaganya, dan ukuran varians tenaga yang memberi petunjuk sama ada keadaan yang dikira membentuk eigenstate Hamiltonian sistem.

Pengguna boleh menentukan litar kuantum yang digunakan dan bilangan shot yang diambil untuk setiap litar kuantum, serta mengawal saiz subspace atau membolehkan penjanaan klasik konfigurasi tambahan untuk membantu konfigurasi yang dijana kuantum. Dengan cara ini pengguna boleh menyesuaikan tingkah laku HI-VQE untuk memenuhi aplikasi yang dikehendaki.
## Pelesenan
Sila ambil perhatian bahawa penggunaan Fungsi Qiskit ini terhad kepada masalah yang memerlukan paling banyak 20 qubit, melainkan lesen diperoleh yang memberikan had yang lebih tinggi.

Sila e-mel [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) jika anda ingin bertanya tentang memperoleh lesen.
## Mulakan
Pertama, [mohon akses ke fungsi](https://forms.office.com/r/zN3hvMTqJ1).
Kemudian, sahkan menggunakan [kunci API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) anda dan, dengan mengandaikan anda telah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan anda, pilih Fungsi Qiskit seperti berikut:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

Pilihan tambahan boleh ditakrifkan dan diberikan untuk sistem molekul dalam format kamus berikut.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Laksanakan fungsi dengan input geometri dan pilihan.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Adalah baik untuk mencetak ID kerja Fungsi supaya ia boleh diberikan dalam permintaan sokongan jika ada yang tidak kena.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


Contoh ini kemudian menggunakan 16 qubit dengan 8 orbital asas sto3g untuk molekul NH3.
Semak [status](/guides/functions#check-job-status) atau kembalikan [keputusan](/guides/functions#retrieve-results) beban kerja Fungsi Qiskit anda seperti berikut:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


Selepas kerja selesai, keputusan boleh diperoleh dengan instans `result()`.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


## Performance

This section shows the demonstrated benchmark calculations of HI-VQE with a 24-qubit case for Li2S, a 40-qubit case for an N2 molecule, and a 44-qubit case for an FeP-NO system.

#### Dissociation potential energy surface curve for an Li2S molecule with 24 qubits

The PES curve is shown with the FCI reference and initial guess from RHF, together with the energy error from the FCI reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

The calculations have been conducted with the following geometries and options.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

The red dots represent the HI-VQE calculation results for six different geometries, and three geometries corresponding to 1.51, 2.40, and 3.80 Angstrom are provided as input in the above cell.

#### Dissociation PES curve for an N2 molecule with 40 qubits

The nitrogen molecule has been identified as a multireference system with large correlation energy contributions beyond the Hartree-Fock state. We conducted a benchmark calculation for the N2 molecule with cc-pvdz basis, (20o,14e) using the homo-lumo active orbital selection. The complete active space (CAS) number to represent this problem is 6,009,350,400. It is not possible to obtain the eigenvalue problem solution (for energy and electronic structure) with this number of states using a powerful desktop (16cpu/64GB). With HI-VQE, users can efficiently search the subspace of CAS states to find chemically accurate results while saving computation resources significantly. The following plots show the PES curve of 40 qubits HI-VQE calculation of the N2 molecule dissociation.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the N2 system](../docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)

#### Dissociation PES curve for five-coordinated iron(II)-porphyrin with an NO system with 44 qubits

Another interesting chemical system is an iron(II)-porphyrin (FeP) complex with a coordinated nitric oxide (NO) ligand, which represents a biologically relevant metalloporphyrin system that plays crucial roles in various physiological processes. In this example, HI-VQE has been utilized to estimate the accurate potential energy surface curve of the intermolecular interaction between FeP and NO (ground state energy for differently separated geometries). The combined system has 450 orbitals and 202 electrons (450o,202e) with 6-31g(d) basis in total. The homo-lumo active orbital selection was utilized to calculate the smaller case from the real case with (22o,22e). From the following benchmark results, we were able to achieve the chemical accuracy (>&nbsp;1.6 mHa) with a state-of-the-art classical computer chemistry calculation of CASCI(DMRG) (22o,22e) reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the FeP-NO system](../docs/images/guides/qunova-chemistry/fepno_44qubits.avif)

## Benchmarks

- Exact matrix size is the number of determinants for exact solution, such as FCI and CASCI.
- HI-VQE calculation samples and calculates the subspace of it (as in, HI-VQE matrix size).
- Total time includes QPU runtime and Qiskit Function runs with CPU.
- Accuracy is estimated from the energy difference from exact solution.

|   Chemical system  | Number of qubits | Exact matrix size |  HI-VQE matrix size  | E(diff) from exact (mHa) | Number of iteration | Total time    | QPU runtime usage |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37 s          | 34 s              |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250 s         | 50 s              |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354 s         | 54 s              |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531 s        | 121 s             |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174 s        | 130 s             |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547 s       | 258 s             |

## Fetch error messages

If your workload fails, the status will be `ERROR` and calling `job.result()` will raise an exception:

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Get support

You can send an email to [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) for help with this function.

If you want help with troubleshooting a specific error, please provide the Function job ID of the job that encountered the error.

## Next steps

<Admonition type="tip" title="Recommendations">
- Request access to the function by filling in [this form](https://forms.office.com/r/zN3hvMTqJ1).
- Visit the [API reference](/docs/api/functions/qunova-chemistry) for this Qiskit Function.
- Try the [Compute dissociation PES curve for FeP-NO with HI-VQE](/docs/tutorials/qunova-hivqe) tutorial.
- Review [Pellow-Jarman, A., et al. (2025).  HIVQE: Handover Iterative Variational Quantum Eigensolver for Efficient Quantum Chemistry Calculations. arXiv preprint arXiv:2503.06292](https://arxiv.org/abs/2503.06292).
- Try the [Dissociation PES curves with Qunova HiVQE](/docs/tutorials/qunova-hivqe) tutorial.
</Admonition>